# Week 4: LLM-Based Similarity Explanations

This notebook adds an explanation layer on top of the patient similarity
model developed in Weeks 1–3. The goal is to translate numeric similarity
into human-readable, clinically grounded explanations using a large
language model (LLM).

This work focuses on interpretability and transparency, not prediction,
diagnosis, or treatment recommendations.



## 4.1 Load similarity model outputs

We reuse the engineered features and similarity logic from Week 3.
No retraining is performed in this step.

In [1]:
import pandas as pd
import numpy as np

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


In [2]:
features = pd.read_csv("//Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/processed/normalized_patient_features_24h_vitals_clean.csv", index_col=0)
labels=features['SepsisLabel']

## 4.2 Select an index patient and nearest neighbors

To illustrate similarity explanations, we begin with a single index patient and their nearest neighbors in the learned feature space.


In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

knn = NearestNeighbors(n_neighbors=6, metric="euclidean")
knn.fit(X_scaled)

index_patient_id = features.index[6]  # this can be changed to any index patient
index_row = features.loc[[index_patient_id]]

distances, indices = knn.kneighbors(
    scaler.transform(index_row),
    n_neighbors=6
)

neighbor_ids = features.index[indices[0][1:]]  # exclude self


## 4.3 Construct structured similarity summaries

Rather than sending raw feature values to an LLM, we first construct
a structured summary describing shared physiological patterns and
key differences between patients.

This reduces hallucination risk and improves interpretability.


In [4]:
def build_similarity_summary(index_id, neighbor_ids, feature_df):
    index_vals = feature_df.loc[index_id]
    neighbor_vals = feature_df.loc[neighbor_ids]

    summary = {
        "index_patient": index_id,
        "num_neighbors": len(neighbor_ids),
        "shared_patterns": {},
        "notable_differences": {}
    }

    for col in feature_df.columns:
        idx_val = index_vals[col]
        neigh_mean = neighbor_vals[col].mean()

        if abs(idx_val - neigh_mean) < 0.25:
            summary["shared_patterns"][col] = (
                "Index patient value is similar to neighbor average"
            )
        else:
            direction = "higher" if idx_val > neigh_mean else "lower"
            summary["notable_differences"][col] = (
                f"Index patient value is {direction} than neighbor average"
            )

    return summary


In [5]:
similarity_summary = build_similarity_summary(
    index_patient_id,
    neighbor_ids,
    features
)


### Why structured summaries?

Numeric similarity alone is difficult to interpret clinically. By converting similarity into structured descriptions of shared and differing features, we create an intermediate representation that can be safely consumed by an LLM.


## 4.4 LLM Prompt Design

The following prompt is designed to generate explanatory text while avoiding diagnosis, prognosis, or treatment recommendations.



In [6]:
PROMPT_TEMPLATE = """
You are a clinical data analyst.

You are given structured information describing similarity between ICU
patients based on physiological measurements.

Your task is to explain in plain language, why the index patient is considered similar to
the comparison patients.

Do NOT:
- Diagnose disease
- Recommend treatment
- Predict outcomes

Focus on:
- Shared physiological patterns
- Observed differences



Structured Input:
{similarity_summary}

Write a concise clinical-style explanation (4–6 sentences).
"""


## 4.5 Example similarity explanation (mocked)

To illustrate the output format, we provide an example explanation generated from the structured summary.


In [7]:
print(PROMPT_TEMPLATE.format(similarity_summary=similarity_summary))



You are a clinical data analyst.

You are given structured information describing similarity between ICU
patients based on physiological measurements.

Your task is to explain in plain language, why the index patient is considered similar to
the comparison patients.

Do NOT:
- Diagnose disease
- Recommend treatment
- Predict outcomes

Focus on:
- Shared physiological patterns
- Observed differences



Structured Input:
{'index_patient': np.int64(9), 'num_neighbors': 5, 'shared_patterns': {'HR_std_24h': 'Index patient value is similar to neighbor average', 'MAP_mean_24h': 'Index patient value is similar to neighbor average', 'MAP_min_24h': 'Index patient value is similar to neighbor average', 'MAP_std_24h': 'Index patient value is similar to neighbor average', 'O2Sat_mean_24h': 'Index patient value is similar to neighbor average', 'O2Sat_min_24h': 'Index patient value is similar to neighbor average', 'O2Sat_max_24h': 'Index patient value is similar to neighbor average', 'O2Sat_std_24h'

### Scope and Safety Considerations

The explanations generated in this notebook are descriptive and observational. They are intended to support interpretability of similarity models and are not intended for clinical decision-making, diagnosis, or treatment.
